In [8]:
#这是要在云端服务器运行的代码，不要考虑本地环境
#下载qwen3.5-4b
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from modelscope import snapshot_download

model_name = "Qwen/Qwen3-4B-Instruct-2507"

print(f"开始下载模型: {model_name}")

model_dir = snapshot_download(model_name)

tokenizer = AutoTokenizer.from_pretrained(
    model_dir,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("模型下载完成！")

save_path = "./Qwen3-4B-Instruct-2507"
tokenizer.save_pretrained(save_path)
model.save_pretrained(save_path)
print(f"模型已保存至: {save_path}")











开始下载模型: Qwen/Qwen3-4B-Instruct-2507


2026-10-15 11:53:10,906 | INFO    | modelscope_hub.download | Downloading 14 files from Qwen/Qwen3-4B-Instruct-2507@master


Downloading:   0%|          | 0/14 [00:00<?, ?file/s]

/usr/local/lib/python3.12/site-packages/modelscope_hub/_download.py:1128: FutureWarning: Environment variable 'INTRA_CLOUD_ACCELERATION_REGION' is deprecated and will be removed in a future version. Use 'MODELSCOPE_DOWNLOAD_INTRA_CLOUD_REGION' instead.
  download_headers = self._build_download_headers(user_agent)


.gitattributes:   0%|          | 0.00/2.22k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

模型下载完成！


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

模型已保存至: ./Qwen3-4B-Instruct-2507


In [9]:
#从intent_train.jsonl中读取数据，分别获取文件中message的role是user和assistant的内容
import json

user_contents = []
assistant_contents = []
with open("intent_train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        messages = data.get("messages", [])
        for msg in messages:
            if msg["role"] == "user":
                user_contents.append(msg["content"])
            elif msg["role"] == "assistant":
                assistant_contents.append(msg["content"])

print(f"共读取 {len(user_contents)} 条user消息, {len(assistant_contents)} 条assistant消息")


共读取 340 条user消息, 340 条assistant消息


In [10]:
#利用qwen3.5-4b模型输入user内容（加上user前缀）生成response
generated_responses = []

for i, user_text in enumerate(user_contents):
    messages = [
        {"role": "user", "content": user_text}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            
            top_p=0.9
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    generated_responses.append(response)

    print(f"\n=== 生成第{i+1}条 ===")
    
    print(f"生成: {response}")
    print(f"真实: {assistant_contents[i] if i < len(assistant_contents) else 'N/A'}")

print(f"\n共生成 {len(generated_responses)} 条回复")


=== 生成第1条 ===
生成: {
  "task": null,
  "knowledge": {
    "intents": [
      "order_info"
    ]
  },
  "chitchat": null
}
真实: {"task": {"commands": [{"command": "start_flow", "flow": "order_status_query"}]}, "knowledge": null, "chitchat": null}

=== 生成第2条 ===
生成: {
  "task": null,
  "knowledge": {
    "intents": [
      "order_info"
    ]
  },
  "chitchat": null
}
真实: {"task": {"commands": [{"command": "start_flow", "flow": "order_status_query"}]}, "knowledge": null, "chitchat": null}

=== 生成第3条 ===
生成: {
  "task": {
    "commands": [
      {
        "command": "start_flow",
        "flow": "order_status_query"
      },
      {
        "command": "set_slots",
        "slots": {
          "order_number": "SO20261001001"
        }
      }
    ]
  },
  "knowledge": null,
  "chitchat": null
}
真实: {"task": {"commands": [{"command": "start_flow", "flow": "order_status_query"}, {"command": "set_slots", "slots": {"order_number": "SO20261001001"}}]}, "knowledge": null, "chitchat": null}

=== 生成

In [13]:
#生成的回复与数据集进行比较，得出准确率（准确率=生成的回复中，与数据集内容相同的占比）
correct_count = 0
total = min(len(generated_responses), len(assistant_contents))

for i in range(total):
    gen = generated_responses[i].strip()
    text= json.loads(gen)
    genf=json.dumps(text, ensure_ascii=False)
    truth = assistant_contents[i]
    if genf == truth:
        correct_count += 1
    else:
        print(f"\n❌ 第{i+1}条不匹配:")
        print(f"  生成: {gen}")
        print(f"  真实: {truth}")

accuracy = correct_count / total if total > 0 else 0.0
print(f"\n========== 评估结果 ==========")
print(f"总样本数: {total}")
print(f"完全匹配数: {correct_count}")
print(f"准确率: {accuracy:.2%}")


❌ 第1条不匹配:
  生成: {
  "task": null,
  "knowledge": {
    "intents": [
      "order_info"
    ]
  },
  "chitchat": null
}
  真实: {"task": {"commands": [{"command": "start_flow", "flow": "order_status_query"}]}, "knowledge": null, "chitchat": null}

❌ 第2条不匹配:
  生成: {
  "task": null,
  "knowledge": {
    "intents": [
      "order_info"
    ]
  },
  "chitchat": null
}
  真实: {"task": {"commands": [{"command": "start_flow", "flow": "order_status_query"}]}, "knowledge": null, "chitchat": null}

❌ 第4条不匹配:
  生成: {
  "task": null,
  "knowledge": {
    "intents": [
      "order_info"
    ]
  },
  "chitchat": null
}
  真实: {"task": {"commands": [{"command": "start_flow", "flow": "order_status_query"}]}, "knowledge": null, "chitchat": null}

❌ 第5条不匹配:
  生成: {
  "task": null,
  "knowledge": {
    "intents": [
      "order_info"
    ]
  },
  "chitchat": null
}
  真实: {"task": {"commands": [{"command": "start_flow", "flow": "order_status_query"}, {"command": "set_slots", "slots": {"order_number": "SO20261